# Module 10 — RAG Optimization Engineering
Google Colab-ready notebook: benchmark → optimize → cache → version → invalidate → regression test.

In [ ]:
import time, random
random.seed(7)
queries=[f'q{i}' for i in range(100)]
configs=[]
for chunk_size in [80,160,320]:
  for k in [3,5,10,20]: configs.append((chunk_size,k))
print('configs:',len(configs))

In [ ]:
def simulated_recall(chunk_size,k):
    base=0.62+0.10*min(k,10)/10
    penalty=abs(chunk_size-160)/1000
    return round(min(.99,base-penalty+random.uniform(-.015,.015)),3)
rows=[]
for cs,k in configs:
    rows.append((cs,k,simulated_recall(cs,k),k*40))
sorted(rows,key=lambda x:(-x[2],x[3]))[:10]

In [ ]:
class LRUCache:
    def __init__(self,capacity=3): self.capacity=capacity; self.data={}
    def get(self,key):
        if key not in self.data: return None
        value=self.data.pop(key); self.data[key]=value; return value
    def put(self,key,value):
        self.data.pop(key,None); self.data[key]=value
        while len(self.data)>self.capacity: self.data.pop(next(iter(self.data)))
cache=LRUCache(3)
for q in ['a','b','c','a','d','a']:
    hit=cache.get(q)
    if hit is None: cache.put(q,'retrieval-'+q)
    print(q,'HIT' if hit else 'MISS',list(cache.data))

## Exercises
1. Replace simulated quality with the Module 9 benchmark.
2. Find the minimum K that meets your Recall@5 target.
3. Add cache-hit/miss timing measurements.
4. Include tenant and policy version in cache keys.
5. Mutate a document and demonstrate stale retrieval.
6. Add index_version and embedding_version to every result.
7. Simulate blue/green index migration and rollback.
8. Build a context-token budget and compare quality/cost.
9. Inject a deletion and write a regression assertion that deleted content is absent.
10. Produce a Pareto-style decision table for quality vs latency vs cost.

In [ ]:
def cache_key(tenant,principal,query,index_version,policy_version):
    return (tenant,principal,query,index_version,policy_version)
print(cache_key('tenant-a','user-1','policy query','idx-2','policy-7'))

## Failure injection
- stale cache after document update
- cache collision across tenants
- mixed embedding/index versions
- deletion not propagated
- context budget removes critical evidence
- new index improves one query class but regresses another

**Gold challenge:** select the cheapest configuration satisfying a quality SLO and security constraints.